# Notebook 5.5: Decoding Strategy Benchmark
So sánh 6 chiến lược giải mã trên mô hình BARTpho Full FT đã train.

**Mục tiêu:** Xác định chiến lược giải mã tối ưu (Greedy, Beam 4, Beam 8, Top-k, Top-p) trước khi áp dụng cho toàn bộ thực nghiệm.

**Thời gian ước tính:** ~1-2 tiếng trên T4 GPU (6 lần inference × 1000 mẫu).

**YÊU CẦU:** BẬT GPU T4 TRÊN KAGGLE

In [ ]:
!pip install -q transformers datasets evaluate rouge_score bert_score sacrebleu peft tqdm "torchao>=0.16.0"

In [ ]:
import pandas as pd
import numpy as np
import torch
import time
import json
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from evaluate import load
from tqdm import tqdm
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Tải dữ liệu & metrics

In [ ]:
# Tải test set
test_df = pd.read_csv("test_1k.csv")
articles = test_df["article"].tolist()
references = test_df["abstract"].tolist()
print(f"Test set: {len(articles)} mẫu")

# Tải metrics
rouge = load("rouge")
bleu = load("bleu")
bertscore = load("bertscore")
print("Đã tải metrics: ROUGE, BLEU, BERTScore")

## 2. Tải mô hình BARTpho FFT (chỉ load 1 lần)

In [ ]:
FULL_FT_PATH = "./bartpho_full_ft_final"

if not os.path.exists(FULL_FT_PATH):
    raise FileNotFoundError(
        f"Không tìm thấy model tại {FULL_FT_PATH}. "
        f"Hãy đảm bảo đã upload/mount checkpoint BARTpho Full FT."
    )

print("Đang tải BARTpho Full FT...")
tokenizer = AutoTokenizer.from_pretrained(FULL_FT_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(FULL_FT_PATH).to(device)
model.eval()
print(f"Đã tải model: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params")

## 3. Định nghĩa 6 chiến lược giải mã

In [ ]:
# Định nghĩa 6 chiến lược giải mã
DECODING_STRATEGIES = {
    "Greedy": dict(
        do_sample=False,
        num_beams=1,
    ),
    "Beam_4": dict(
        do_sample=False,
        num_beams=4,
        early_stopping=True,
    ),
    "Beam_8": dict(
        do_sample=False,
        num_beams=8,
        early_stopping=True,
    ),
    "Top_k_50": dict(
        do_sample=True,
        top_k=50,
        temperature=1.0,
    ),
    "Top_p_0.9": dict(
        do_sample=True,
        top_p=0.9,
        top_k=0,  # Disable top-k when using top-p
        temperature=1.0,
    ),
    "Top_p_0.95": dict(
        do_sample=True,
        top_p=0.95,
        top_k=0,
        temperature=1.0,
    ),
}

print("Các chiến lược giải mã:")
for name, params in DECODING_STRATEGIES.items():
    print(f"  {name}: {params}")

## 4. Hàm inference & đánh giá

In [ ]:
def generate_summaries(model, tokenizer, texts, gen_kwargs, strategy_name):
    """Sinh tóm tắt cho danh sách văn bản với cấu hình giải mã tùy ý."""
    summaries = []
    for text in tqdm(texts, desc=f"Generating ({strategy_name})"):
        inputs = tokenizer(
            text, max_length=1024, truncation=True, return_tensors="pt"
        ).to(device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=128,
                **gen_kwargs
            )
        
        summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
        summaries.append(summary)
    return summaries


def evaluate_summaries(predictions, refs):
    """Tính ROUGE, BLEU, BERTScore."""
    actual_refs = refs[:len(predictions)]
    
    r_score = rouge.compute(predictions=predictions, references=actual_refs)
    bleu_score = bleu.compute(predictions=predictions, references=actual_refs)
    bert_result = bertscore.compute(
        predictions=predictions, references=actual_refs, lang="vi"
    )
    
    return {
        "ROUGE-1": round(r_score['rouge1'] * 100, 2),
        "ROUGE-2": round(r_score['rouge2'] * 100, 2),
        "ROUGE-L": round(r_score['rougeL'] * 100, 2),
        "ROUGE-Lsum": round(r_score['rougeLsum'] * 100, 2),
        "BLEU": round(bleu_score['bleu'] * 100, 2),
        "BERT-F1": round(np.mean(bert_result['f1']) * 100, 2),
    }


print("Đã định nghĩa hàm generate_summaries() và evaluate_summaries()")

## 5. Chạy benchmark 6 chiến lược
Model chỉ được load 1 lần, chạy inference 6 lần với các `gen_kwargs` khác nhau.

In [ ]:
all_results = {}
all_predictions = {}  # Lưu predictions để phân tích thêm nếu cần

for strategy_name, gen_kwargs in DECODING_STRATEGIES.items():
    print(f"\n{'='*60}")
    print(f"CHIẾN LƯỢC: {strategy_name}")
    print(f"Params: {gen_kwargs}")
    print(f"{'='*60}")
    
    # Đo thời gian
    start_time = time.time()
    
    # Sinh tóm tắt
    predictions = generate_summaries(
        model, tokenizer, articles, gen_kwargs, strategy_name
    )
    
    gen_time = time.time() - start_time
    
    # Đánh giá
    print(f"\nĐang tính metrics...")
    metrics = evaluate_summaries(predictions, references)
    metrics["Time (min)"] = round(gen_time / 60, 1)
    
    all_results[strategy_name] = metrics
    all_predictions[strategy_name] = predictions
    
    print(f"\nKẾT QUẢ {strategy_name}:")
    for k, v in metrics.items():
        print(f"  {k}: {v}")
    print(f"  Thời gian: {gen_time/60:.1f} phút")

print(f"\n\n{'='*60}")
print("ĐÃ HOÀN THÀNH BENCHMARK TẤT CẢ 6 CHIẾN LƯỢC!")
print(f"{'='*60}")

## 6. Tổng hợp & hiển thị kết quả

In [ ]:
# Tạo bảng tổng hợp
results_df = pd.DataFrame(all_results).T
results_df.index.name = "Strategy"

# Sắp xếp theo ROUGE-1 giảm dần
results_df = results_df.sort_values("ROUGE-1", ascending=False)

print("\n" + "="*80)
print("BẢNG TỔNG HỢP KẾT QUẢ BENCHMARK DECODING STRATEGIES")
print("Model: BARTpho Full FT | Test set: 1,000 mẫu")
print("="*80)
print(results_df.to_string())
print("="*80)

# Highlight chiến lược tốt nhất
best_rouge1 = results_df["ROUGE-1"].idxmax()
best_bleu = results_df["BLEU"].idxmax()
print(f"\n🏆 Tốt nhất ROUGE-1: {best_rouge1} ({results_df.loc[best_rouge1, 'ROUGE-1']})")
print(f"🏆 Tốt nhất BLEU:    {best_bleu} ({results_df.loc[best_bleu, 'BLEU']})")

## 7. Lưu kết quả

In [ ]:
# Lưu bảng kết quả ra CSV
results_df.to_csv("decoding_strategy_benchmark.csv")
print("Đã lưu: decoding_strategy_benchmark.csv")

# Lưu predictions ra CSV để kiểm tra định tính
pred_df = pd.DataFrame({
    "article": articles,
    "reference": references,
})
for strategy_name, preds in all_predictions.items():
    pred_df[f"pred_{strategy_name}"] = preds

pred_df.to_csv("decoding_strategy_predictions.csv", index=False)
print("Đã lưu: decoding_strategy_predictions.csv")

# Lưu kết quả dạng JSON (dễ đọc lại)
with open("decoding_strategy_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)
print("Đã lưu: decoding_strategy_results.json")

print("\n✅ Hoàn tất! Copy bảng kết quả ở cell trên vào paper.")

## 8. Xem ví dụ đầu ra (Qualitative Analysis)

In [ ]:
# Hiển thị 3 ví dụ so sánh giữa các chiến lược
SAMPLE_INDICES = [0, 50, 100]

for idx in SAMPLE_INDICES:
    print(f"\n{'='*80}")
    print(f"VÍ DỤ #{idx}")
    print(f"{'='*80}")
    print(f"\n📝 ARTICLE (100 ký tự đầu): {articles[idx][:100]}...")
    print(f"\n🎯 REFERENCE: {references[idx]}")
    print()
    for strategy_name, preds in all_predictions.items():
        print(f"  [{strategy_name}]: {preds[idx]}")
    print()

In [ ]:
# Giải phóng bộ nhớ GPU
del model
torch.cuda.empty_cache()
print("Đã giải phóng GPU memory.")